# Sistema RSA para firmas digitales

En este notebook, mostraremos el proceso para generar las llaves pública y privada, así como el proceso para generar una firma digital y su verificación correspondiente usando el sistema RSA. Empezaremos por cargar las funciones que necesitamos:

In [ ]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_encryption,
    rsa_decryption,
    sha256_of_sentence,
)

## Protocolo RSA para la generación de llaves

Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras llaves *pública* y *privada.*

In [ ]:
(public_key, private_key) = rsa_key_generation(1000)
d = private_key
e = public_key[0]
n = public_key[1]

print(f"Llave privada (d): {d}\n")
print(f"Llave pública (e): {e}\n")
print(f"Módulo para las llaves (n): {n}\n")

Recordemos que el valor $d$ de la llave privada se debe de mantener en secreto. En cambio los valores de $e$ y $n$ corresponden a la llave pública y son conocidos por todos los agentes involucrados, incluída Eva.

## Protocolo RSA para la firma de mensajes

Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [ ]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Para generar la firma digital $s$, Alicia debe de utilizar el mismo algoritmo que utiliza para descifrar mensajes. En otras palabras, Alicia debe calcular
$$
s = h^{d} \mod n
$$
y ya con esta información puede generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m, s)
$$

In [ ]:
s = rsa_decryption(private_key, h)
signed_message = (m, s)

print(f"El mensaje firmado es: {signed_message}\n")

Para verificar la firma, Beto aplica ahora el mismo algoritmo que utilizaría para encriptar mensajes. De manera más precisa, Beto calcula el Hash
$$
h = \operatorname{Hash}(m)
$$
y calcula también
$$
\tilde{h} = s^{e} \mod n
$$
Si estos dos números son iguales, entonces la firma es válida.

In [ ]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]
h_tilde = rsa_encryption(public_key, s)

if h == h_tilde:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de h es: {h}\n")
print(f"El valor de h̃ es: {h_tilde}\n")
print("Por lo tanto,", verificacion_firma)